# 01 - Data Exploration
Exploratory analysis for Vaani Kavach voice dataset.

In [1]:
# Cell 1: Dependencies installation
!pip install -q transformers datasets torchaudio scikit-learn librosa soundfile tqdm pyyaml

In [6]:
import json
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import roc_curve
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from tqdm.auto import tqdm

# Configuration
MODEL_NAME = "MattyB95/AST-ASVspoof2019-Synthetic-Voice-Detection"
DATASET_NAME = "SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA"
# Configuration update
DATASET_SPLIT_NAME = "test"   # Hugging Face dataset split ka actual naam
CONTRACT_SPLIT_NAME = "eval"  # Contract JSON require karta hai "eval"
MAX_SAMPLES = 500  # Set to None for full split evaluation
OUTPUT_PATH = "baseline_results.json"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# 1. Load Model & Extractor
print(f"Loading model: {MODEL_NAME}...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = ASTForAudioClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

# Check Model Label Mapping
# id2label usually maps {0: 'bonafide', 1: 'spoof'} or similar
id2label = model.config.id2label
print(f"Model id2label mapping: {id2label}")

# Determine spoof index from model config
spoof_idx = 1
for idx, lbl in id2label.items():
    if "spoof" in str(lbl).lower() or "fake" in str(lbl).lower():
        spoof_idx = int(idx)
        break

# Stream dataset using the repo's split name ('test')
print(f"Connecting to Hugging Face dataset stream ({DATASET_SPLIT_NAME} split)...")
dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT_NAME, streaming=True)

# 3. Helper for EER
def compute_eer(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    eer_threshold_idx = np.nanargmin(np.absolute((fnr - fpr)))
    eer = fpr[eer_threshold_idx]
    return float(eer)

# 4. Evaluation Loop
ground_truth = []
spoof_scores = []
predictions = []

count = 0
with torch.no_grad():
    for item in tqdm(dataset, desc="Evaluating", total=MAX_SAMPLES):
        # Handle audio extraction safely
        audio_data = item.get("audio", item.get("audio_file", None))
        if audio_data is None:
            continue

        audio_array = audio_data["array"]
        sampling_rate = audio_data["sampling_rate"]

        # Handle label extraction (0 = bonafide, 1 = spoof)
        raw_label = item.get("label", item.get("key", item.get("tag", None)))

        if isinstance(raw_label, str):
            label = 1 if "spoof" in raw_label.lower() else 0
        elif isinstance(raw_label, int):
            label = 1 if raw_label == 1 else 0
        else:
            # Fallback check
            label = 1 if item.get("is_spoof", False) else 0

        # Preprocess via AST Feature Extractor
        inputs = feature_extractor(
            audio_array,
            sampling_rate=sampling_rate,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

        spoof_prob = float(probs[spoof_idx])
        pred_label = int(np.argmax(probs))

        # Standardize prediction: if predicted index matches spoof_idx -> 1, else 0
        pred_binary = 1 if pred_label == spoof_idx else 0

        ground_truth.append(label)
        spoof_scores.append(spoof_prob)
        predictions.append(pred_binary)

        count += 1
        if MAX_SAMPLES and count >= MAX_SAMPLES:
            break

# 5. Metrics Calculation
ground_truth = np.array(ground_truth)
predictions = np.array(predictions)
spoof_scores = np.array(spoof_scores)

accuracy = float(np.mean(predictions == ground_truth))
eer_val = compute_eer(ground_truth, spoof_scores)

# 6. Save strict JSON contract artifact
baseline_data = {
    "model_name": MODEL_NAME,
    "dataset_split": CONTRACT_SPLIT_NAME,
    "accuracy": round(accuracy, 4),
    "eer": round(eer_val, 4)
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(baseline_data, f, indent=2)

print("\n--- Evaluation Finished ---")
print(json.dumps(baseline_data, indent=2))

Running on: cuda
Loading model: MattyB95/AST-ASVspoof2019-Synthetic-Voice-Detection...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Model id2label mapping: {0: 'Bonafide', 1: 'Spoof'}
Connecting to Hugging Face dataset stream (test split)...


Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]


--- Evaluation Finished ---
{
  "model_name": "MattyB95/AST-ASVspoof2019-Synthetic-Voice-Detection",
  "dataset_split": "eval",
  "accuracy": 0.91,
  "eer": 0.0508
}


### Baseline Evaluation Summary & Handoff Notes

- **Model Evaluated:** `MattyB95/AST-ASVspoof2019-Synthetic-Voice-Detection` (Pretrained Audio Spectrogram Transformer)

- **Dataset & Split:** `SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA` (streamed in-memory, `test` split mapped to contract `eval`)

- **Evaluation Metrics:**
  - **Accuracy:** `91.0%`
  - **Equal Error Rate (EER):** `5.08%`

- **Artifact Generated:** Exported to `ml_pipeline/notebooks/baseline_results.json` adhering to the team contract schema (`model_name`, `dataset_split`, `accuracy`, `eer`).

- **Next Steps & Handoffs:**
  - **Devyani (`architecture.py`, `train.py`):** Baseline target benchmark is established. Custom fine-tuning (wav2vec2/AASIST-style) can now proceed with the goal of improving generalization and beating the 5.08% EER baseline.
  
  - **Tanvi (`inference`):** Baseline model inference is verified; streaming buffer and sliding-window wrapper integration can safely hook into the pipeline.
  
  - **Backend/Frontend Team:** Static baseline contract is ready to unblock end-to-end integration without blocking on final fine-tuned weights.